# Round 2: aligning V1, V4 and IT at once (monkey N)

| | targets (area-layer: weight) | sum | from |
|---|---|---|---|
| pretrained | none (epoch 0) | 0 | — |
| control | none, CE only | 0 | Round 1 |
| IT only w1 | IT-layer8: 1 | 1 | Round 1 |
| 3 areas, w1 each | V1-layer2: 1, V4-layer6: 1, IT-layer8: 1 | 3 | Round 2 |
| 3 areas, w1/3 each | V1-layer2: 1/3, V4-layer6: 1/3, IT-layer8: 1/3 | 1 | Round 2 |

Shared: monkey N, val split for evaluation, batches of 64 from one recording day (same batches in every run), 10 epochs,
AdamW lr 3e-5 cosine, fp32, CE on pseudo-labels (steps 70-99), CKA_S loss = mean over the target's bins of log(1 − CKA_S).
Target bins / response windows: V1 20-200 ms (bins 2-19), V4 50-200 ms (5-19), IT 70-200 ms (7-19).

Inputs: `Round1/outputs/round1/{control, student_IT_layer8_w1}` (incl. `accuracy.npz`), `Round2/outputs/{multi3_w1each, multi3_w0p33each}`,
`prepare_images/things_imagenet_overlap.csv`, raw monkey N files in `~/Desktop/neural_data` (Figure 7).

In [ ]:
import csv
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)

CN = Path("~/Desktop/DynVision/CordsNet").expanduser()
NEURAL = Path("~/Desktop/neural_data").expanduser()
run_dir = {"control": CN / "Round1/outputs/round1/control",
           "IT only w1": CN / "Round1/outputs/round1/student_IT_layer8_w1",
           "3 areas, w1 each": CN / "Round2/outputs/multi3_w1each",
           "3 areas, w1/3 each": CN / "Round2/outputs/multi3_w0p33each"}
runs = list(run_dir)
round2 = runs[2:]
color = {"pretrained": "0.35", "control": "#2a78d6", "IT only w1": "#1baf7a",
         "3 areas, w1 each": "#8e44ad", "3 areas, w1/3 each": "#eb6834"}
targets = [("V1", "layer2", np.arange(2, 20)), ("V4", "layer6", np.arange(5, 20)), ("IT", "layer8", np.arange(7, 20))]
window = {area: bins for area, _, bins in targets}

train = {r: pd.read_csv(run_dir[r] / "train_metrics.csv") for r in runs}
ev = {r: [np.load(f) for f in sorted((run_dir[r] / "eval").glob("epoch_*.npz"))] for r in runs}
n_epochs = min(len(ev[r]) for r in runs) - 1
r0 = ev["control"][0]
layers, areas = [str(x) for x in r0["layers"]], [str(x) for x in r0["areas"]]
S = 1                                                              # CKA_S in cka_pooled[..., estimator]
t_ms = r0["bin_start_ms"] + 5
steps_per_epoch = int(train["control"].groupby("epoch")["step"].max().iloc[0])

# CKA_S per target over its window: (run, target) -> array over epochs; "pretrained" = epoch 0
tgt = {(r, a): np.array([np.nanmean(e["cka_pooled"][layers.index(l), areas.index(a), b, S]) for e in ev[r][:n_epochs + 1]])
       for r in runs for a, l, b in targets}
for a, l, b in targets:
    tgt[("pretrained", a)] = np.full(n_epochs + 1, tgt[("control", a)][0])

print(f"{n_epochs} epochs, {steps_per_epoch} steps per epoch")
print("epoch 0 identical across runs:",
      all(np.allclose(ev[r][0]["cka_pooled"], r0["cka_pooled"], equal_nan=True) for r in runs))

## 1. Training curves

Per step, smoothed over 50 steps. Train CKA_S per target = mean over the target's bins (per batch).
Round 1 runs logged only IT-layer8.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for r in runs:
    d = train[r]
    x = d["global_step"] / steps_per_epoch
    axes[0].plot(x, d["loss_ce"].rolling(50, min_periods=1).mean(), color=color[r], lw=2, label=r)
    for ax, (a, l, b) in zip(axes[1:], targets):
        cols = [f"cka_S_{a}_{l}_bin{k:02d}" for k in b]
        if not set(cols) <= set(d.columns):
            cols = [f"cka_S_bin{k:02d}" for k in b] if a == "IT" else None     # Round 1 column names
        if cols:
            ax.plot(x, d[cols].mean(axis=1).rolling(50, min_periods=1).mean(), color=color[r], lw=2)
axes[0].set(yscale="log", title="train CE", ylabel="CE")
for ax, (a, l, b) in zip(axes[1:], targets):
    ax.set(title=f"train CKA_S, {a} vs {l}")
for ax in axes:
    ax.set(xlabel="epoch", xticks=range(n_epochs + 1))
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Val CKA_S per target over epochs

Dashed: val (pooled over batches, after each epoch). Solid: train (mean over the epoch's steps), where logged.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (a, l, b) in zip(axes, targets):
    ax.axhline(tgt[("pretrained", a)][0], color=color["pretrained"], lw=1, ls=":", label="pretrained")
    for r in runs:
        ax.plot(range(n_epochs + 1), tgt[(r, a)], "o--", color=color[r], lw=2, ms=4, label=r)
        cols = [f"cka_S_{a}_{l}_bin{k:02d}" for k in b]
        if not set(cols) <= set(train[r].columns):
            cols = [f"cka_S_bin{k:02d}" for k in b] if a == "IT" else None
        if cols:
            tr = train[r].assign(c=train[r][cols].mean(axis=1)).groupby("epoch")["c"].mean()
            ax.plot(tr.index, tr.values, "-", color=color[r], lw=1.5, alpha=0.7)
    ax.set(title=f"{a} vs {l} ({b[0] * 10}-200 ms)", xlabel="epoch", xticks=range(n_epochs + 1))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("CKA_S")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"val CKA_S at epoch {n_epochs} (change from pretrained):")
for a, l, b in targets:
    print(f"  {a}-{l}: pretrained {tgt[('pretrained', a)][0]:.3f}   "
          + "   ".join(f"{r} {tgt[(r, a)][-1]:.3f} ({tgt[(r, a)][-1] - tgt[(r, a)][0]:+.3f})" for r in runs))

## 3. IT: does adding V1 and V4 targets change the IT alignment?

Gain of IT-layer8 CKA_S over pretrained. IT weight: 1 in "IT only w1" and "w1 each", 1/3 in "w1/3 each".

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.axhline(0, color="0.5", lw=1)
for r in runs:
    y = tgt[(r, "IT")]
    ax.plot(range(n_epochs + 1), y - y[0], "o-", color=color[r], lw=2, ms=4, label=r)
ax.set(title="IT vs layer8: gain over pretrained", xlabel="epoch", ylabel="Δ CKA_S", xticks=range(n_epochs + 1))
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. When does each target improve?

CKA_S per 10 ms bin at epoch 10. Gray: the target's bins.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (a, l, b) in zip(axes, targets):
    i, j = layers.index(l), areas.index(a)
    ax.axvspan(b[0] * 10, 200, color="0.92", zorder=0)
    ax.axhline(0, color="0.5", lw=1)
    ax.plot(t_ms, r0["cka_pooled"][i, j, :, S], "o-", color=color["pretrained"], lw=2, ms=3, label="pretrained")
    for r in runs:
        ax.plot(t_ms, ev[r][n_epochs]["cka_pooled"][i, j, :, S], "o-", color=color[r], lw=2, ms=3, label=r)
    ax.set(title=f"{a} vs {l}, epoch {n_epochs}", xlabel="time after image onset (ms)", xlim=(0, 200))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("CKA_S")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. All layers and areas: spread of the alignment

Change in CKA_S (response window) from pretrained to epoch 10. Boxes: the run's alignment targets.

In [ ]:
win = {r: np.array([[np.nanmean(e["cka_pooled"][i, j, window[a], S]) for j, a in enumerate(areas)] for i in range(len(layers))])
       for r, e in [(r, ev[r][n_epochs]) for r in runs] + [("pretrained", r0)]}
lim = max(np.nanmax(np.abs(win[r] - win["pretrained"])) for r in runs)
boxes = {"control": [], "IT only w1": [("IT", "layer8")], "3 areas, w1 each": [(a, l) for a, l, _ in targets],
         "3 areas, w1/3 each": [(a, l) for a, l, _ in targets]}

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, r in zip(axes.flat, runs):
    d = win[r] - win["pretrained"]
    im = ax.imshow(d.T, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    for i in range(len(layers)):
        for j in range(len(areas)):
            ax.text(i, j, f"{d[i, j]:+.2f}", ha="center", va="center", fontsize=8, color="w" if abs(d[i, j]) > 0.6 * lim else "k")
    for a, l in boxes[r]:
        ax.add_patch(plt.Rectangle((layers.index(l) - 0.5, areas.index(a) - 0.5), 1, 1, fill=False, ec="k", lw=2))
    ax.set(title=r, xticks=range(8), xticklabels=range(1, 9), yticks=range(len(areas)), yticklabels=areas)
for ax in axes[1]:
    ax.set_xlabel("model layer")
fig.colorbar(im, ax=axes, label="Δ CKA_S vs pretrained", shrink=0.9)
plt.show()

print("pretrained CKA_S (response window), layer1 ... layer8:")
for j, a in enumerate(areas):
    print(f"  {a}: " + "  ".join(f"{v:.3f}" for v in win["pretrained"][:, j]))

## 6. Accuracy

Same val images for every run: the 1,728 images of the evaluation batches. Agreement: prediction = pseudo-label.
THINGS: images of the 548 concepts matched to ImageNet; correct if the prediction is one of the matched classes.
Round 1 runs: from `accuracy.npz` (recomputed from the checkpoints); Round 2 runs: from the per-epoch evaluation.
Top: at 200 ms per epoch. Bottom: per time step at epoch 10 (gray: CE loss steps).

In [ ]:
overlap = {r["uniqueID"]: [int(i) for i in r["imagenet_idx"].split(";")]
           for r in csv.DictReader(open(CN / "prepare_images/things_imagenet_overlap.csv"))}
acc_r1 = {r: np.load(run_dir[r] / "accuracy.npz") for r in runs[:2]}
a1 = acc_r1["control"]
pos = {int(s): k for k, s in enumerate(a1["stim_idx"])}                       # stim_idx -> index in accuracy.npz
eval_stim = ev[round2[0]][0]["eval_rows"] + 1                                  # stim_idx of the evaluation images
idx = np.array([pos[int(s)] for s in eval_stim])
plabel, concept = a1["pseudo_label"][idx], a1["concept"][idx]
match = np.zeros((len(idx), 1000), dtype=bool)
for k, c in enumerate(concept):
    if c in overlap:
        match[k, overlap[c]] = True
in_ov = match.any(axis=1)

# top5[run]: (epoch, image, step, 5) for the same images in the same order
top5 = {r: acc_r1[r]["top5"][:n_epochs + 1][:, idx].astype(int) for r in runs[:2]}
for r in round2:
    assert all(np.array_equal(e["eval_rows"], ev[round2[0]][0]["eval_rows"]) for e in ev[r]), "evaluation images differ"
    top5[r] = np.stack([e["top5"] for e in ev[r][:n_epochs + 1]]).astype(int)
agree, top1, top5acc = {}, {}, {}
for r in runs:
    hit = match[np.arange(len(idx))[None, :, None, None], top5[r]]
    agree[r] = (top5[r][..., 0] == plabel[None, :, None]).mean(axis=1)              # (epoch, step)
    top1[r] = hit[..., 0][:, in_ov].mean(axis=1)
    top5acc[r] = hit[:, in_ov].any(axis=-1).mean(axis=1)
print(f"{len(idx)} images, {in_ov.sum()} in matched concepts; pretrained agreement at 200 ms: {agree['control'][0, -1]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (name, d) in zip(axes, [("agreement with pseudo-labels", agree), ("THINGS top-1", top1), ("THINGS top-5", top5acc)]):
    ax.axhline(d["control"][0, -1], color=color["pretrained"], lw=1, ls="--", label="pretrained")
    for r in runs:
        ax.plot(range(n_epochs + 1), d[r][:, -1], "o-", color=color[r], lw=2, ms=4, label=r)
    ax.set(title=f"{name}, 200 ms", xlabel="epoch", xticks=range(n_epochs + 1))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("fraction of val images")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

t_step = (np.arange(top5["control"].shape[2]) + 1) * 2
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (name, d) in zip(axes, [("agreement with pseudo-labels", agree), ("THINGS top-1", top1)]):
    ax.axvspan(141, 201, color="0.92", zorder=0)
    ax.plot(t_step, d["control"][0], "-", color=color["pretrained"], lw=2, label="pretrained")
    for r in runs:
        ax.plot(t_step, d[r][n_epochs], "-", color=color[r], lw=2, label=r)
    ax.set(title=f"{name}, epoch {n_epochs}", xlabel="time after image onset (ms)", xlim=(0, 200))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("fraction of val images")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"at 200 ms, epoch {n_epochs} (THINGS n = {in_ov.sum()}):")
print(f"  {'pretrained':20s} agreement {agree['control'][0, -1]:.3f}  top-1 {top1['control'][0, -1]:.3f}  top-5 {top5acc['control'][0, -1]:.3f}")
for r in runs:
    print(f"  {r:20s} agreement {agree[r][-1, -1]:.3f}  top-1 {top1[r][-1, -1]:.3f}  top-5 {top5acc[r][-1, -1]:.3f}")

## 7. Layer dynamics vs the brain

Mean activity of each target layer over time (val images, epoch 10), with monkey N's matching area (black dashed).
Top: scaled to first bin = 0, peak = 1. Bottom: unscaled.

In [ ]:
neural_tc = {}
for a, l, b in targets:
    total, n = np.zeros(20), 0
    for f in sorted(NEURAL.rglob(f"monkeyN_day*_{a}_binned.npz")):
        mua = np.load(f)["binned_mua"]
        if a == "V1":
            mua = np.delete(mua, np.arange(320, 384), axis=2)
        total += mua.sum(axis=(1, 2))
        n += mua.shape[1] * mua.shape[2]
    neural_tc[a] = total / n if n else None

fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharex=True)
for col, (a, l, b) in enumerate(targets):
    i = layers.index(l)
    for r, e in [("pretrained", r0)] + [(r, ev[r][n_epochs]) for r in runs]:
        c = e["model_timecourse"][i]
        axes[0, col].plot(t_ms, (c - c[0]) / (c.max() - c[0]), "o-", color=color[r], lw=2, ms=3, label=r)
        axes[1, col].plot(t_ms, c, "o-", color=color[r], lw=2, ms=3)
    if neural_tc[a] is not None:
        c = neural_tc[a]
        axes[0, col].plot(t_ms, (c - c[0]) / (c.max() - c[0]), "o--", color="k", lw=2, ms=3, label=f"monkey N {a}")
    axes[0, col].set(title=f"{l} (target: {a}), normalized")
    axes[1, col].set(title=f"{l}, absolute", xlabel="time after image onset (ms)")
    for ax in axes[:, col]:
        ax.set_xlim(0, 200)
        ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("normalized activity")
axes[1, 0].set_ylabel("mean activity")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()